In [1]:
import pandas as pd
import numpy as np
from collections import Counter

# ============================================================
# 0. 데이터 로드
# ============================================================
df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번 파일(최종)\M19_도매_소매업(최종).parquet')
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}컬럼")


# ============================================================
# 1. Dickinson 생애주기 분류 함수
# ============================================================
def classify_lifecycle(ocf_sign, icf_sign, fcf_sign):
    """
    Dickinson(2011) 현금흐름 부호 조합으로 생애주기 분류
      성숙기: +/-/-
      성장기: +/-/+
      도입기: -/-/+
      쇠퇴기: -/+/- 또는 -/+/+ (변형 포함)
      조정기: +/+/-, -/-/-, +/+/+
    """
    pattern = f"{ocf_sign}/{icf_sign}/{fcf_sign}"
    lifecycle_map = {
        '+/-/-': '성숙기',
        '+/-/+': '성장기',
        '-/-/+': '도입기',
        '-/+/-': '쇠퇴기',
        '-/+/+': '쇠퇴기',
        '+/+/-': '조정기',
        '-/-/-': '조정기',
        '+/+/+': '조정기',
    }
    return lifecycle_map.get(pattern, '조정기')


# ============================================================
# 2. 각 행에 생애주기 분류 적용
# ============================================================
df['OCF_부호'] = df['영업현금흐름비율'].apply(lambda x: '+' if x > 0 else '-')
df['ICF_부호'] = df['투자활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['FCF_부호'] = df['재무활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')

df['생애주기_단년'] = df.apply(
    lambda row: classify_lifecycle(row['OCF_부호'], row['ICF_부호'], row['FCF_부호']),
    axis=1
)


# ============================================================
# 3. 최근 3년 다수결로 생애주기 확정
#    - 3년 모두 다른 패턴이면 마지막 행 사용
#    - 3년 미만 데이터면 마지막 행 사용
# ============================================================
def get_majority_lifecycle(group):
    group_sorted = group.sort_values('회계년도')
    recent = group_sorted.tail(3)
    last_lifecycle = group_sorted.iloc[-1]['생애주기_단년']

    if len(recent) < 3:
        return last_lifecycle

    counter = Counter(recent['생애주기_단년'].tolist())
    most_common = counter.most_common()

    if most_common[0][1] >= 2:
        return most_common[0][0]
    else:
        return last_lifecycle  # 3개 모두 다른 경우 → 마지막 행


lifecycle_result = df.groupby('사업자등록번호').apply(
    get_majority_lifecycle
).reset_index()
lifecycle_result.columns = ['사업자등록번호', '생애주기_최종']

print(f"\n=== 생애주기 분류 완료 ===")
print(lifecycle_result['생애주기_최종'].value_counts())


# ============================================================
# 4. 마지막 행 추출 + 생애주기 병합
# ============================================================
last_rows = df.sort_values('회계년도').groupby('사업자등록번호').last().reset_index()
last_rows = last_rows.merge(lifecycle_result, on='사업자등록번호', how='left')

total_bad  = int(last_rows['부실라벨_ICR3년'].sum())
total_good = int((last_rows['부실라벨_ICR3년'] == 0).sum())
print(f"\n=== 마지막 행 기준 데이터 ===")
print(f"전체 기업 수: {len(last_rows):,}  |  부실: {total_bad:,}  |  정상: {total_good:,}")


# ============================================================
# 5. WoE 계산
# ============================================================
def calculate_woe(df, category_col, label_col):
    total_bad  = df[label_col].sum()
    total_good = (df[label_col] == 0).sum()

    rows = []
    for cat in df[category_col].unique():
        sub  = df[df[category_col] == cat]
        bad  = sub[label_col].sum()
        good = (sub[label_col] == 0).sum()

        # 라플라스 스무딩 (0 나누기 방지)
        bad_rate  = (bad  + 0.5) / (total_bad  + 0.5)
        good_rate = (good + 0.5) / (total_good + 0.5)

        woe = np.log(bad_rate / good_rate)
        iv  = (bad_rate - good_rate) * woe

        rows.append({
            '생애주기':  cat,
            '전체':      len(sub),
            '부실':      int(bad),
            '정상':      int(good),
            '부실률(%)': round(bad / len(sub) * 100, 2),
            '부실비율':  round(bad_rate, 4),
            '정상비율':  round(good_rate, 4),
            'WoE':       round(woe, 4),
            'IV':        round(iv, 4),
        })

    result = pd.DataFrame(rows).sort_values('WoE', ascending=False).reset_index(drop=True)
    result['IV_합계'] = round(result['IV'].sum(), 4)
    return result

woe_table = calculate_woe(last_rows, '생애주기_최종', '부실라벨_ICR3년')

print("\n=== WoE 계산 결과 ===")
print(woe_table.to_string(index=False))
print(f"\nIV 합계: {woe_table['IV_합계'].iloc[0]}  (0.3 이상 = 강한 변별력)")


# ============================================================
# 6. PDO 변환
#    β=1 (동등 가중), 기준점수=500, PDO=20, 기준오즈=부실/정상
#
#    공식:
#      factor = PDO / ln(2)
#      offset = 기준점수 - factor × ln(기준오즈)
#      점수   = offset + factor × WoE   (β=1이므로 WoE 그대로)
#      부호   = WoE 양수일수록 위험 → 점수 높을수록 위험 방향 유지
# ============================================================
BASE_SCORE = 500
PDO        = 20
BASE_ODDS  = total_bad / total_good   # 실제 부실률 기반

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

print(f"\n=== PDO 파라미터 ===")
print(f"기준점수 : {BASE_SCORE}")
print(f"PDO      : {PDO}")
print(f"기준오즈 : {BASE_ODDS:.4f}  ({total_bad}/{total_good})")
print(f"factor   : {factor:.4f}")
print(f"offset   : {offset:.4f}")

# WoE → PDO 점수
woe_table['PDO_점수'] = (offset + factor * woe_table['WoE']).round(2)

print("\n=== PDO 변환 후 점수 ===")
print(woe_table[['생애주기', '부실률(%)', 'WoE', 'PDO_점수']].sort_values('PDO_점수', ascending=False).to_string(index=False))


# ============================================================
# 7. 0~100 스케일링 (위험 스코어: 높을수록 위험)
# ============================================================
pdo_min = woe_table['PDO_점수'].min()
pdo_max = woe_table['PDO_점수'].max()

woe_table['위험점수_최종'] = (
    (woe_table['PDO_점수'] - pdo_min) / (pdo_max - pdo_min) * 100
).round(1)

print("\n=== 생애주기별 최종 위험 점수 (0~100) ===")
final = woe_table[['생애주기', '전체', '부실', '부실률(%)', 'WoE', 'PDO_점수', '위험점수_최종']]
print(final.sort_values('위험점수_최종', ascending=False).to_string(index=False))


# ============================================================
# 8. 기업별 점수 부여 및 저장
# ============================================================
score_map = dict(zip(woe_table['생애주기'], woe_table['위험점수_최종']))
last_rows['생애주기_점수'] = last_rows['생애주기_최종'].map(score_map)

print("\n=== 기업별 점수 샘플 ===")
print(last_rows[['사업자등록번호', '회계년도', '생애주기_최종', '생애주기_점수', '부실라벨_ICR3년']].head(10).to_string(index=False))

last_rows.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\생애주기 스코어링/lifecycle_scored_pdo.parquet', index=False)

# WoE/점수 테이블도 저장
woe_table.to_csv(r'C:\유비온프로젝트2\corporate-bankruptcy\생애주기 스코어링/lifecycle_woe_table.csv', index=False, encoding='utf-8-sig')

print("\n✅ 저장 완료")
print("   - lifecycle_scored_pdo.parquet : 기업별 점수")
print("   - lifecycle_woe_table.csv      : 생애주기별 WoE/점수 테이블")

전체 데이터: 39,908행 × 277컬럼

=== 생애주기 분류 완료 ===
생애주기_최종
성숙기    2004
조정기    1194
도입기    1143
쇠퇴기     984
성장기     799
Name: count, dtype: int64

=== 마지막 행 기준 데이터 ===
전체 기업 수: 6,124  |  부실: 1,509  |  정상: 4,615

=== WoE 계산 결과 ===
생애주기   전체  부실   정상  부실률(%)   부실비율   정상비율     WoE     IV  IV_합계
 도입기 1143 490  653   42.87 0.3249 0.1416  0.8307 0.1523 0.4674
 쇠퇴기  984 375  609   38.11 0.2488 0.1321  0.6333 0.0739 0.4674
 조정기 1194 267  927   22.36 0.1772 0.2010 -0.1257 0.0030 0.4674
 성장기  799 153  646   19.15 0.1017 0.1401 -0.3202 0.0123 0.4674
 성숙기 2004 224 1780   11.18 0.1487 0.3858 -0.9531 0.2259 0.4674

IV 합계: 0.4674  (0.3 이상 = 강한 변별력)

=== PDO 파라미터 ===
기준점수 : 500
PDO      : 20
기준오즈 : 0.3270  (1509/4615)
factor   : 28.8539
offset   : 532.2548

=== PDO 변환 후 점수 ===
생애주기  부실률(%)     WoE  PDO_점수
 도입기   42.87  0.8307  556.22
 쇠퇴기   38.11  0.6333  550.53
 조정기   22.36 -0.1257  528.63
 성장기   19.15 -0.3202  523.02
 성숙기   11.18 -0.9531  504.75

=== 생애주기별 최종 위험 점수 (0~100) ===
생애주기   전체  부실  부실률(%)     WoE 